# transformer-inference-lab — GQA training run

Trains the GQA checkpoint (`n_kv_head=2`) for the memory–latency–quality comparison against MHA and MQA. Run as **Save & Run All (Commit)** — ~2.4-2.8h on a T4 (GQA should be slightly faster than MHA due to smaller KV projections).

**Before running:** attach both `transformer-inference-lab-data` and `transformer-inference-lab-checkpoints` (sidebar → Add Input). The second isn't strictly needed for this run but keeping it attached means you can reference the MHA checkpoint later in the same session if you want a quick comparison without a separate notebook.

Confirm `configs/gqa.yaml` has `max_iters: 5000` on GitHub before running cell 7, and confirm the val_loss fix (final checkpoint now computes it automatically) is in train.py — check the git log in cell 1.

## 1. GPU check + clone repo

In [ ]:
!nvidia-smi
!git clone https://github.com/modestesavadogo/transformer-inference-lab.git
%cd /kaggle/working/transformer-inference-lab
!git log --oneline -8

## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt --quiet

## 3. Link the dataset
Requires `transformer-inference-lab-data` attached via the notebook's Input sidebar.

Note the mount path includes `/datasets/<username>/` — confirmed from the MHA run, not the shorter path you'd expect from the sidebar name alone.

In [ ]:
!mkdir -p data
!ln -s /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data/train.bin data/train.bin
!ln -s /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data/val.bin data/val.bin
!ls -la data/
!test -f data/train.bin && echo "train.bin target exists" || echo "train.bin target MISSING"
!test -f data/val.bin && echo "val.bin target exists" || echo "val.bin target MISSING"

## 4. Verify data before committing to a multi-hour run

In [ ]:
import numpy as np

train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')
print(f"train: {len(train_data):,} tokens")
print(f"val:   {len(val_data):,} tokens")

assert len(train_data) > 40_000_000, "train.bin looks too small — check dataset link"
assert len(val_data) > 10_000, "val.bin looks too small — check dataset link"
print("data check passed")

## 5. Confirm config is set for the real run
Confirm `n_kv_head: 2` (this is what makes it GQA, not MHA), `max_iters: 5000`, `batch_size: 8`, `grad_accum_steps: 8`. All fields except `n_kv_head` should be identical to `configs/mha.yaml`.

In [ ]:
!cat configs/gqa.yaml

## 6. Run tests — cheap insurance before a multi-hour job

In [ ]:
!python -m pytest tests/ -q

## 7. Train GQA
Runs to completion in the background under Save & Run All, even if the tab is closed.

In [ ]:
!python train.py --config configs/gqa.yaml --device cuda --eval-interval 250 --log-interval 50

## 8. Confirm the checkpoint is valid
val_loss should be populated automatically now — no backfill needed this time.

In [ ]:
import torch

ckpt = torch.load("results/checkpoints/gqa.pt", map_location="cuda")
print("iter:", ckpt["iter"])
print("val_loss:", ckpt.get("val_loss"))
print("config:", ckpt["config"])
print("num tensors in state dict:", len(ckpt["model_state_dict"]))

assert ckpt.get("val_loss") is not None, "val_loss missing — train.py fix may not have been pulled"

## 9. Upload checkpoint as a Kaggle Dataset
Same pattern as MHA — do this before the session recycles, don't rely on Output alone.

In [ ]:
!mkdir -p checkpoint_upload
!cp results/checkpoints/gqa.pt checkpoint_upload/

In [ ]:
import json

metadata = {
    "title": "transformer-inference-lab-checkpoints",
    "id": "modestesavadogomaths/transformer-inference-lab-checkpoints",
    "licenses": [{"name": "CC0-1.0"}]
}
with open("checkpoint_upload/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(open("checkpoint_upload/dataset-metadata.json").read())

In [ ]:
# NOTE: this dataset already exists (created for mha.pt) — use 'version' not 'create'
# to add gqa.pt into the same dataset rather than erroring on a duplicate.
!kaggle datasets version -p checkpoint_upload/ -m "add gqa.pt checkpoint" --dir-mode zip